In [1]:
from typing import Dict, List, TypedDict, Any, Literal, Annotated, Optional, cast
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
import operator
import argparse
import sys

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_community.embeddings import SentenceTransformerEmbeddings

In [2]:
import dotenv

dotenv.load_dotenv()

True

In [3]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [4]:
from langfuse.langchain import CallbackHandler


langfuse_handler = CallbackHandler()

In [90]:
# 1. 메모장(State) 확장: 모든 단계의 결과물을 저장할 수 있어야 합니다.
class MyRAGState(TypedDict):
    # 최초 입력
    query: str
    # 제안 요청서 양식에 맞춰 개선한 입력
    dense_query: str
    # dense_query에서 핵심 키워드들만 추출한 입력
    sparse_query: str
    hierarchy_info: List[dict]  # 2-1 결과
    retrieved_docs: Annotated[List[dict], operator.add] # 2-2~2-5 결과 합산
    reranked_context: List[dict] # 2-6 결과
    answer: str

In [91]:
# from hybrid_search_v1.search_graph import graph as step2_graph

In [101]:
import numpy as np
import copy
import json
import sqlite3
import sqlite_vec
from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from collections import Counter


DB_PATH = "/home/codeitDev/project/AI_7-team/DB/document.db"
MODEL_NAME = "jhgan/ko-sroberta-multitask"

embeddings = SentenceTransformerEmbeddings(model_name=MODEL_NAME)

kiwi = Kiwi()

class SearchState(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    scoped_dense_result: List[Dict]
    scoped_sparse_result: List[Dict]
    dense_result: List[Dict]
    sparse_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def hierarchy_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(h.metadata, '$.doc_id'),
                json_extract(h.metadata, '$.level'),
                json_extract(h.metadata, '$.title'),
                json_extract(p.metadata, '$.level'),
                json_extract(p.metadata, '$.title'),
                v.distance
            FROM hierarchy h
            LEFT JOIN hierarchy p
            ON json_extract(h.metadata, '$.parent_uid')
                = json_extract(p.metadata, '$.uid')
            JOIN hierarchy_vec v ON h.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 100
        """, (query_vec,))
        results = cursor.fetchall()

        # 성적의 합을 통해, 가장 좋은 문서 50개를 반환한다.
        scores = {}
        counts = {}
        for row in results:
            if counts.get(row[0], 0) == 3:
                continue
            else:
                counts[row[0]] = counts.get(row[0], 0) + 1
                score = 1.0 / (1.0 + row[5])
                scores[row[0]] = scores.get(row[0], 0) + score
        
        sorted_docs = sorted(scores.items(), key=lambda x: -x[1])
        search_results = [doc_id for doc_id, score in sorted_docs[:30]]

    return {'scopes': search_results}


def extract_nouns(query):
    if not query:
        return ""
    tokens: List[Any] = cast(List[Any], kiwi.tokenize(query))
    nouns = [f'"{t.form}"' for t in tokens if t.tag in ('NNG', 'NNP', 'NNB')]
    if len(nouns) == 0:
        return ""
    return " OR ".join(nouns)


def scoped_dense_search(state: SearchState):
    scopes = state['scopes']
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    results = {}
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        # 각 scope (계층 정보) 마다
        scope_query = f"""
            WITH scoped_chunks AS (
                SELECT *
                FROM chunks
                WHERE json_extract(metadata, '$.doc_id') IN (SELECT value FROM json_each(?))
            )
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata,
                v.distance
            FROM scoped_chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """
        cursor = conn.cursor()
        cursor.execute(scope_query, (json.dumps(list(scopes)), query_vec,))
        results = cursor.fetchall()
        results = [row for row in results]
        results = sorted(results, key=lambda x: x[3])
        
        # 'uid': ('text', 'metadata') 형식이다.
        results = [{result[0]:(result[1], json.loads(result[2]))} for result in results]

    return {'scoped_dense_result': results}


def scoped_sparse_search(state: SearchState):
    scopes = state['scopes']
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'scoped_sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                doc_id,
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ? AND doc_id IN (SELECT value FROM json_each(?))
                ORDER BY score
                LIMIT 30
        """, (query, json.dumps(list(scopes)),))

        sparse_result = cursor.fetchall()
        uids = [r[1] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    if len(final_results) >= 30:
        final_results = final_results[:30]
    return {'scoped_sparse_result': final_results}


def dense_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def sparse_search(state: SearchState):
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ?
                ORDER BY score
                LIMIT 30
        """, (query,))

        sparse_result = cursor.fetchall()
        uids = [r[0] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    return {'sparse_result': final_results}


def rrf(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    scoped_dense_scores = compute_scores(state['scoped_dense_result'])
    scoped_sparse_scores = compute_scores(state['scoped_sparse_result'])
    dense_scores = compute_scores(state['dense_result'])
    sparse_scores = compute_scores(state['sparse_result'],)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['scoped_dense_result'],
        state['scoped_sparse_result'],
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores,
        scoped_dense_scores,
        scoped_sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [102]:
search_workflow = StateGraph(SearchState)

search_workflow.add_node("hierarchy", hierarchy_search)

search_workflow.add_node("scoped_dense", scoped_dense_search)
search_workflow.add_node("scoped_sparse", scoped_sparse_search)
search_workflow.add_node("dense", dense_search)
search_workflow.add_node("sparse", sparse_search)

search_workflow.add_node("rrf", rrf)

search_workflow.set_entry_point("hierarchy")

search_workflow.add_edge("hierarchy", "scoped_dense")
search_workflow.add_edge("hierarchy", "scoped_sparse")
search_workflow.add_edge("hierarchy", "dense")
search_workflow.add_edge("hierarchy", "sparse")

search_workflow.add_edge("scoped_dense", "rrf")
search_workflow.add_edge("scoped_sparse", "rrf")
search_workflow.add_edge("dense", "rrf")
search_workflow.add_edge("sparse", "rrf")
search_workflow.add_edge("rrf", END)

In [103]:
hybrid_app = search_workflow.compile()

In [104]:
class SearchState2(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    dense_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def rrf2(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'])

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}


search_workflow2 = StateGraph(SearchState2)

search_workflow2.add_node("dense", dense_search)

search_workflow2.add_node("rrf", rrf2)

search_workflow2.set_entry_point("dense")
search_workflow2.add_edge("dense", "rrf")
search_workflow2.add_edge("rrf", END)

In [105]:
dense_app = search_workflow2.compile()

In [106]:
class SearchState3(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    scoped_dense_result: List[Dict]
    dense_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def rrf3(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    scoped_dense_scores = compute_scores(state['scoped_dense_result'])
    dense_scores = compute_scores(state['dense_result'])

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['scoped_dense_result'],
        state['dense_result'],
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        scoped_dense_scores,
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

search_workflow3 = StateGraph(SearchState3)

search_workflow3.add_node("hierarchy", hierarchy_search)

search_workflow3.add_node("scoped_dense", scoped_dense_search)
search_workflow3.add_node("dense", dense_search)

search_workflow3.add_node("rrf", rrf3)

search_workflow3.set_entry_point("hierarchy")

search_workflow3.add_edge("hierarchy", "scoped_dense")
search_workflow3.add_edge("hierarchy", "dense")

search_workflow3.add_edge("scoped_dense", "rrf")
search_workflow3.add_edge("dense", "rrf")
search_workflow3.add_edge("rrf", END)

In [107]:
dense_with_hierarchy_app = search_workflow3.compile()

In [108]:
import os
import re
import random
from kiwipiepy import Kiwi

kiwi = Kiwi()


def split_paragraphs(md_text):
    # 빈 줄 기준 분리
    paragraphs = re.split(r"\n\s*\n", md_text)
    # 너무 짧은 문단 제거
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 200]
    return paragraphs


def split_sentences(paragraph):
    # 간단한 문장 분리 (과도한 regex 지양)
    sentences = re.split(r"(?<=[.!?다요])\s+", paragraph)
    return [s.strip() for s in sentences if len(s.strip()) > 30]


def sanitize_for_fts5(query: str) -> str:
    # 허용되는 문자만 남기고 제거
    return re.sub(r"[^가-힣a-zA-Z0-9\s]", "", query).strip()


def generate_queries_from_markdown(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    dense_queries = []
    sparse_queries = []
    hybrid_queries = []

    while len(dense_queries) < 2 or \
          len(sparse_queries) < 2 or \
          len(hybrid_queries) < 2:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(dense_queries) < 2:
            start = random.randint(0, max(1, len(s)//4))
            end = random.randint(len(s)//2, len(s))
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            dense_queries.append({
                "query": q,
                "original": s
            })

        # ---- Sparse ----
        if len(sparse_queries) < 2:
            nouns = extract_nouns(s)
            if not nouns:
                continue
            nouns = nouns.strip().split()
            # if len(nouns) >= 5:
            # selected = nouns[:7]
            q = " ".join(nouns)
            q = sanitize_for_fts5(q)
            if len(q) >= 3:
                sparse_queries.append({
                    "query": q,
                    "original": s
                })

        # ---- Hybrid ----
        if len(hybrid_queries) < 2:
            nouns = extract_nouns(s)
            if not nouns:
                continue
            nouns = nouns.strip().split()
            # if len(nouns) >= 3:
            # selected = nouns[:3]
            
            '''start = random.randint(0, max(1, len(s)//3))
            end = min(len(s), start + 60)
            base = s[start:end]'''
            q = s + " " + " ".join(nouns)
            q = sanitize_for_fts5(q)
            if len(q) >= 3:
                hybrid_queries.append({
                    "query": q,
                    "original": s
                })

    return {
        "dense": dense_queries[:2],
        "sparse": sparse_queries[:2],
        "hybrid": hybrid_queries[:2],
    }

In [109]:
def generate_all_queries(folder_path):
    results = {}

    for file in os.listdir(folder_path):
        if file.endswith(".md") and file.startswith("step2"):
            path = os.path.join(folder_path, file)
            results[file] = generate_queries_from_markdown(path)

    return results

In [110]:
def run_self_retrieval_test(md_path, retriever):
    queries = generate_queries_from_markdown(md_path)

    stats = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "sparse": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["dense", "sparse", "hybrid"]:

        for item in queries[q_type]:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retriever.invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [111]:
def run_full_evaluation(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "sparse": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [112]:
def print_results(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["dense", "sparse", "hybrid"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [83]:
aggregate = run_full_evaluation('/home/codeitDev/project/AI_7-team/output', app)
print_results(aggregate)

{'dense': {'hit': 2, 'total': 2, 'ranks': [1, 1]}, 'sparse': {'hit': 2, 'total': 2, 'ranks': [1, 1]}, 'hybrid': {'hit': 2, 'total': 2, 'ranks': [2, 1]}}
{'dense': {'hit': 4, 'total': 4, 'ranks': [1, 1, 4, 7]}, 'sparse': {'hit': 4, 'total': 4, 'ranks': [1, 1, 27, 6]}, 'hybrid': {'hit': 4, 'total': 4, 'ranks': [2, 1, 4, 10]}}
{'dense': {'hit': 6, 'total': 6, 'ranks': [1, 1, 4, 7, 3, 1]}, 'sparse': {'hit': 6, 'total': 6, 'ranks': [1, 1, 27, 6, 16, 15]}, 'hybrid': {'hit': 6, 'total': 6, 'ranks': [2, 1, 4, 10, 2, 19]}}
{'dense': {'hit': 8, 'total': 8, 'ranks': [1, 1, 4, 7, 3, 1, 3, 6]}, 'sparse': {'hit': 8, 'total': 8, 'ranks': [1, 1, 27, 6, 16, 15, 2, 9]}, 'hybrid': {'hit': 8, 'total': 8, 'ranks': [2, 1, 4, 10, 2, 19, 3, 8]}}
{'dense': {'hit': 10, 'total': 10, 'ranks': [1, 1, 4, 7, 3, 1, 3, 6, 1, 23]}, 'sparse': {'hit': 10, 'total': 10, 'ranks': [1, 1, 27, 6, 16, 15, 2, 9, 5, 16]}, 'hybrid': {'hit': 10, 'total': 10, 'ranks': [2, 1, 4, 10, 2, 19, 3, 8, 29, 16]}}
{'dense': {'hit': 12, 'total

KeyboardInterrupt: 

In [113]:
import os
import re
import random
from kiwipiepy import Kiwi

kiwi = Kiwi()


def split_paragraphs(md_text):
    # 빈 줄 기준 분리
    paragraphs = re.split(r"\n\s*\n", md_text)
    # 너무 짧은 문단 제거
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 200]
    return paragraphs


def split_sentences(paragraph):
    # 간단한 문장 분리 (과도한 regex 지양)
    sentences = re.split(r"(?<=[.!?다요])\s+", paragraph)
    return [s.strip() for s in sentences if len(s.strip()) > 30]


def sanitize_for_fts5(query: str) -> str:
    # 허용되는 문자만 남기고 제거
    return re.sub(r"[^가-힣a-zA-Z0-9\s]", "", query).strip()


def generate_queries_from_markdown(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    queries = []
    sparse_queries = []
    hybrid_queries = []

    while len(queries) < 2:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(queries) < 2:
            start = random.randint(0, max(1, len(s)//4))
            end = random.randint(len(s)//2, len(s))
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            queries.append({
                "query": q,
                "original": s
            })

    return queries[:2]

In [125]:
def generate_all_queries(folder_path):
    results = {}

    for file in os.listdir(folder_path):
        if file.endswith(".md") and file.startswith("step2"):
            path = os.path.join(folder_path, file)
            results[file] = generate_queries_from_markdown(path)

    return results

In [115]:
def run_self_retrieval_test(md_path, retrievers):
    queries = generate_queries_from_markdown(md_path)

    stats = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "dense_with_hierarchy": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["dense", "dense_with_hierarchy", "hybrid"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [116]:
def run_full_evaluation(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "dense_with_hierarchy": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [117]:
def print_results(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["dense", "dense_with_hierarchy", "hybrid"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [118]:
aggregate = run_full_evaluation('/home/codeitDev/project/AI_7-team/output', {"dense": dense_app, "dense_with_hierarchy": dense_with_hierarchy_app, "hybrid": hybrid_app})
print_results(aggregate)

{'dense': {'hit': 2, 'total': 2, 'ranks': [1, 1]}, 'dense_with_hierarchy': {'hit': 2, 'total': 2, 'ranks': [1, 1]}, 'hybrid': {'hit': 2, 'total': 2, 'ranks': [1, 1]}}
{'dense': {'hit': 3, 'total': 4, 'ranks': [1, 1, 1]}, 'dense_with_hierarchy': {'hit': 3, 'total': 4, 'ranks': [1, 1, 12]}, 'hybrid': {'hit': 4, 'total': 4, 'ranks': [1, 1, 1, 22]}}
{'dense': {'hit': 5, 'total': 6, 'ranks': [1, 1, 1, 1, 29]}, 'dense_with_hierarchy': {'hit': 5, 'total': 6, 'ranks': [1, 1, 12, 11, 29]}, 'hybrid': {'hit': 6, 'total': 6, 'ranks': [1, 1, 1, 22, 4, 11]}}
{'dense': {'hit': 7, 'total': 8, 'ranks': [1, 1, 1, 1, 29, 1, 1]}, 'dense_with_hierarchy': {'hit': 7, 'total': 8, 'ranks': [1, 1, 12, 11, 29, 12, 17]}, 'hybrid': {'hit': 8, 'total': 8, 'ranks': [1, 1, 1, 22, 4, 11, 2, 1]}}
{'dense': {'hit': 8, 'total': 10, 'ranks': [1, 1, 1, 1, 29, 1, 1, 1]}, 'dense_with_hierarchy': {'hit': 8, 'total': 10, 'ranks': [1, 1, 12, 11, 29, 12, 17, 6]}, 'hybrid': {'hit': 10, 'total': 10, 'ranks': [1, 1, 1, 22, 4, 11, 2

In [121]:
import numpy as np
import copy
import json
import sqlite3
import sqlite_vec
from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from collections import Counter


DB_PATH = "/home/codeitDev/project/AI_7-team/DB/document.db"
MODEL_NAME = "jhgan/ko-sroberta-multitask"

embeddings = SentenceTransformerEmbeddings(model_name=MODEL_NAME)

kiwi = Kiwi()

class SearchState4(TypedDict):
    # 검색용 쿼리
    query: str

    dense_result: List[Dict]
    sparse_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def extract_nouns(query):
    if not query:
        return ""
    tokens: List[Any] = cast(List[Any], kiwi.tokenize(query))
    nouns = [f'"{t.form}"' for t in tokens if t.tag in ('NNG', 'NNP', 'NNB')]
    if len(nouns) == 0:
        return ""
    return " OR ".join(nouns)


def dense_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def sparse_search(state: SearchState):
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ?
                ORDER BY score
                LIMIT 30
        """, (query,))

        sparse_result = cursor.fetchall()
        uids = [r[0] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    return {'sparse_result': final_results}


def rrf4(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'])
    sparse_scores = compute_scores(state['sparse_result'],)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [122]:
def empty(SearchState):
    return

In [135]:
search_workflow4 = StateGraph(SearchState)

search_workflow4.add_node("empty", empty)

search_workflow4.add_node("dense", dense_search)
search_workflow4.add_node("sparse", sparse_search)

search_workflow4.add_node("rrf", rrf4)

search_workflow4.set_entry_point("empty")

search_workflow4.add_edge("empty", "dense")
search_workflow4.add_edge("empty", "sparse")

search_workflow4.add_edge("dense", "rrf")
search_workflow4.add_edge("rrf", END)

In [136]:
hybrid_app2 = search_workflow4.compile()

In [150]:
def run_self_retrieval_test2(md_path, retrievers):
    queries = generate_queries_from_markdown(md_path)

    stats = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid2"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [ ]:
def run_full_evaluation2(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test2(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [152]:
def print_results2(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["hybrid2"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [ ]:
aggregate = run_full_evaluation2('/home/codeitDev/project/AI_7-team/output', {"hybrid2": hybrid_app2})
print_results2(aggregate)

{'hybrid2': {'hit': 2, 'total': 2, 'ranks': [1, 4]}}
{'hybrid2': {'hit': 4, 'total': 4, 'ranks': [1, 4, 1, 1]}}
{'hybrid2': {'hit': 6, 'total': 6, 'ranks': [1, 4, 1, 1, 1, 5]}}
{'hybrid2': {'hit': 7, 'total': 8, 'ranks': [1, 4, 1, 1, 1, 5, 1]}}
{'hybrid2': {'hit': 9, 'total': 10, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1]}}
{'hybrid2': {'hit': 11, 'total': 12, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1]}}
{'hybrid2': {'hit': 13, 'total': 14, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1]}}
{'hybrid2': {'hit': 14, 'total': 16, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1, 7]}}
{'hybrid2': {'hit': 16, 'total': 18, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1, 7, 1, 1]}}
{'hybrid2': {'hit': 18, 'total': 20, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1, 7, 1, 1, 2, 2]}}
{'hybrid2': {'hit': 20, 'total': 22, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1, 7, 1, 1, 2, 2, 1, 14]}}
{'hybrid2': {'hit': 22, 'total': 24, 'ranks': [1, 4, 1, 1, 1, 5, 1, 2, 1, 1, 1, 1, 1, 7, 1, 1, 2, 2, 1, 14, 2

In [141]:
def rrf5(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list, w = 1):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'], 3)
    sparse_scores = compute_scores(state['sparse_result'], 2)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [144]:
search_workflow5 = StateGraph(SearchState)

search_workflow5.add_node("empty", empty)

search_workflow5.add_node("dense", dense_search)
search_workflow5.add_node("sparse", sparse_search)

search_workflow5.add_node("rrf", rrf5)

search_workflow5.set_entry_point("empty")

search_workflow5.add_edge("empty", "dense")
search_workflow5.add_edge("empty", "sparse")

search_workflow5.add_edge("dense", "rrf")
search_workflow5.add_edge("sparse", "rrf")
search_workflow5.add_edge("rrf", END)

In [145]:
hybrid_app3 = search_workflow5.compile()

In [ ]:
def run_self_retrieval_test3(md_path, retrievers):
    queries = generate_queries_from_markdown(md_path)

    stats = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid3"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [ ]:
def run_full_evaluation3(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test3(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [161]:
def print_results3(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["hybrid3"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [ ]:
aggregate = run_full_evaluation3('/home/codeitDev/project/AI_7-team/output', {"hybrid3": hybrid_app3})
print_results3(aggregate)

{'hybrid3': {'hit': 2, 'total': 2, 'ranks': [1, 3]}}
{'hybrid3': {'hit': 4, 'total': 4, 'ranks': [1, 3, 1, 3]}}
{'hybrid3': {'hit': 6, 'total': 6, 'ranks': [1, 3, 1, 3, 1, 3]}}
{'hybrid3': {'hit': 7, 'total': 8, 'ranks': [1, 3, 1, 3, 1, 3, 1]}}
{'hybrid3': {'hit': 9, 'total': 10, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1]}}
{'hybrid3': {'hit': 11, 'total': 12, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6]}}
{'hybrid3': {'hit': 13, 'total': 14, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4]}}
{'hybrid3': {'hit': 14, 'total': 16, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4, 1]}}
{'hybrid3': {'hit': 16, 'total': 18, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4, 1, 1, 2]}}
{'hybrid3': {'hit': 18, 'total': 20, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4, 1, 1, 2, 1, 4]}}
{'hybrid3': {'hit': 20, 'total': 22, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4, 1, 1, 2, 1, 4, 1, 5]}}
{'hybrid3': {'hit': 22, 'total': 24, 'ranks': [1, 3, 1, 3, 1, 3, 1, 1, 1, 1, 6, 1, 4, 1, 1, 2, 1, 4, 1, 5, 1, 

In [153]:
import os
import re
import random
from kiwipiepy import Kiwi

kiwi = Kiwi()


def generate_queries_from_markdown2(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    queries = []
    sparse_queries = []
    hybrid_queries = []

    while len(queries) < 10:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(queries) < 10:
            start = random.randint(0, max(1, len(s)//4))
            end = random.randint(len(s)//2, len(s))
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            queries.append({
                "query": q,
                "original": s
            })

    return queries[:10]

In [154]:
def run_self_retrieval_test4(md_path, retrievers):
    queries = generate_queries_from_markdown2(md_path)

    stats = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid2"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [155]:
def run_full_evaluation4(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test4(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [156]:
aggregate = run_full_evaluation4('/home/codeitDev/project/AI_7-team/output', {"hybrid2": hybrid_app2})
print_results2(aggregate)

{'hybrid2': {'hit': 8, 'total': 10, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1]}}
{'hybrid2': {'hit': 18, 'total': 20, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4]}}
{'hybrid2': {'hit': 28, 'total': 30, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4, 8, 1, 2, 1, 4, 7, 1, 1, 1, 1]}}
{'hybrid2': {'hit': 38, 'total': 40, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4, 8, 1, 2, 1, 4, 7, 1, 1, 1, 1, 2, 1, 1, 3, 10, 2, 1, 5, 1, 1]}}
{'hybrid2': {'hit': 48, 'total': 50, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4, 8, 1, 2, 1, 4, 7, 1, 1, 1, 1, 2, 1, 1, 3, 10, 2, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1]}}
{'hybrid2': {'hit': 58, 'total': 60, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4, 8, 1, 2, 1, 4, 7, 1, 1, 1, 1, 2, 1, 1, 3, 10, 2, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 3, 1, 3, 5, 1, 1, 1, 1, 1]}}
{'hybrid2': {'hit': 68, 'total': 70, 'ranks': [3, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 19, 2, 1, 7, 4, 8, 

In [157]:
def run_self_retrieval_test5(md_path, retrievers):
    queries = generate_queries_from_markdown2(md_path)

    stats = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid3"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [158]:
def run_full_evaluation5(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test5(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [163]:
for i in range(3):
    aggregate = run_full_evaluation5('/home/codeitDev/project/AI_7-team/output', {"hybrid3": hybrid_app3})
    print_results3(aggregate)

{'hybrid3': {'hit': 8, 'total': 10, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2]}}
{'hybrid3': {'hit': 18, 'total': 20, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1]}}
{'hybrid3': {'hit': 27, 'total': 30, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1, 1, 12, 8, 1, 4, 1, 1, 1, 4]}}
{'hybrid3': {'hit': 36, 'total': 40, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1, 1, 12, 8, 1, 4, 1, 1, 1, 4, 5, 24, 1, 5, 2, 1, 10, 1, 1]}}
{'hybrid3': {'hit': 45, 'total': 50, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1, 1, 12, 8, 1, 4, 1, 1, 1, 4, 5, 24, 1, 5, 2, 1, 10, 1, 1, 1, 1, 1, 9, 3, 1, 1, 1, 1]}}
{'hybrid3': {'hit': 50, 'total': 60, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1, 1, 12, 8, 1, 4, 1, 1, 1, 4, 5, 24, 1, 5, 2, 1, 10, 1, 1, 1, 1, 1, 9, 3, 1, 1, 1, 1, 30, 4, 1, 3, 1]}}
{'hybrid3': {'hit': 60, 'total': 70, 'ranks': [1, 2, 1, 1, 10, 2, 1, 2, 2, 2, 1, 19, 1, 1, 1, 1, 1, 1, 1, 12, 8, 1, 4, 1, 1, 1, 4, 5,